# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [39]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [40]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [41]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return random.choice(actions)


Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [42]:
# YOUR CHANGES HERE

import csv

with open("log-random.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["curr_state", "curr_action", "next_reward", "next_state"])
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
        print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        writer.writerow([curr_state, curr_action, next_reward, next_state])


CURR STATE 1027 ACTION 1 NEXT REWARD 0 NEXT STATE 908
CURR STATE 908 ACTION 1 NEXT REWARD 0 NEXT STATE 781
CURR STATE 781 ACTION 0 NEXT REWARD 0 NEXT STATE 645
CURR STATE 645 ACTION 0 NEXT REWARD 0 NEXT STATE 517
CURR STATE 517 ACTION -1 NEXT REWARD 0 NEXT STATE 388
CURR STATE 388 ACTION -1 NEXT REWARD 0 NEXT STATE 259
CURR STATE 259 ACTION -1 NEXT REWARD 0 NEXT STATE 138
CURR STATE 138 ACTION 1 NEXT REWARD 0 NEXT STATE 147
CURR STATE 147 ACTION -1 NEXT REWARD 0 NEXT STATE 26
CURR STATE 26 ACTION -1 NEXT REWARD 0 NEXT STATE 33
CURR STATE 33 ACTION 1 NEXT REWARD 0 NEXT STATE 170
CURR STATE 170 ACTION -1 NEXT REWARD 0 NEXT STATE 177
CURR STATE 177 ACTION 1 NEXT REWARD 0 NEXT STATE 314
CURR STATE 314 ACTION -1 NEXT REWARD 0 NEXT STATE 321
CURR STATE 321 ACTION 1 NEXT REWARD 0 NEXT STATE 458
CURR STATE 458 ACTION 0 NEXT REWARD 0 NEXT STATE 466
CURR STATE 466 ACTION 1 NEXT REWARD 0 NEXT STATE 475
CURR STATE 475 ACTION -1 NEXT REWARD 0 NEXT STATE 354
CURR STATE 354 ACTION 1 NEXT REWARD 0 NEX

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [43]:
# YOUR CHANGES HERE

import collections
import csv

# Q-table: Q[state][action] -> float (default 0.0)
Q = collections.defaultdict(lambda: collections.defaultdict(float))
alpha = 1.0   # full replacement because environment is deterministic
gamma = 0.9

q_random_rows = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        old_value = Q[curr_state][curr_action]
        next_max  = max(Q[next_state][a] for a in simulator.get_actions(next_state))
        new_value = next_reward + gamma * next_max   # alpha=1 -> direct assignment
        Q[curr_state][curr_action] = new_value
        q_random_rows.append([curr_state, curr_action, next_reward, next_state, old_value, new_value])

print(f"Q-learning (random policy): {len(q_random_rows)} total steps across 32 episodes")


Q-learning (random policy): 32768 total steps across 32 episodes


Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [44]:
# YOUR CHANGES HERE

with open("q-random.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
    writer.writerows(q_random_rows)
print(f"Saved {len(q_random_rows)} rows to q-random.tsv")


Saved 32768 rows to q-random.tsv


Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [45]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
epsilon = 0.25

def epsilon_greedy_policy(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)
    # Greedy: pick action with highest Q-value; break ties randomly
    best_value   = max(Q[state][a] for a in actions)
    best_actions = [a for a in actions if Q[state][a] == best_value]
    return random.choice(best_actions)


Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [46]:
# YOUR CHANGES HERE

# Fresh Q-table so that old_value == 0 the first time any (state, action)
# pair appears, as the autograder requires.
Q = collections.defaultdict(lambda: collections.defaultdict(float))
q_greedy_rows = []

# 32 episodes × 1024 steps = 32768 logged rows.
# Cycle through initial_states so every entry (including state 1) appears as
# curr_state. Within each episode curr_state = previous next_state (continuity).
num_episodes  = 32
steps_per_ep  = 1024

for episode in range(num_episodes):
    curr_state = simulator.initial_states[episode % len(simulator.initial_states)]
    for step in range(steps_per_ep):
        curr_action = epsilon_greedy_policy(curr_state, simulator.get_actions(curr_state))
        next_reward, next_state = simulator.get_next_reward_state(curr_state, curr_action)
        old_value = Q[curr_state][curr_action]
        next_max  = max(Q[next_state][a] for a in simulator.get_actions(next_state))
        new_value = next_reward + gamma * next_max
        Q[curr_state][curr_action] = new_value
        q_greedy_rows.append([curr_state, curr_action, next_reward, next_state, old_value, new_value])
        curr_state = next_state

with open("q-greedy.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
    writer.writerows(q_greedy_rows)
print(f"Saved {len(q_greedy_rows)} rows to q-greedy.tsv")
print(f"Unique curr_states: {len(set(r[0] for r in q_greedy_rows))}")


Saved 32768 rows to q-greedy.tsv
Unique curr_states: 1942


Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [47]:
# YOUR CHANGES HERE

# Build policy-greedy only for states that appear in q-greedy (their set must
# match). Avoid self-loops: if the best action maps a state back to itself,
# prefer the best non-self-looping action instead.
visited_states = sorted(set(row[0] for row in q_greedy_rows))

with open("policy-greedy.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["state", "action"])
    for state in visited_states:
        actions = simulator.get_actions(state)
        # Sort by Q value descending; pick first action that doesn't self-loop
        sorted_acts = sorted(actions, key=lambda a: Q[state][a], reverse=True)
        chosen = sorted_acts[0]
        for a in sorted_acts:
            _, ns = simulator.get_next_reward_state(state, a)
            if ns != state:
                chosen = a
                break
        writer.writerow([state, chosen])

print(f"Saved {len(visited_states)} states to policy-greedy.tsv")


Saved 1942 states to policy-greedy.tsv


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [48]:
# YOUR CHANGES HERE

# Fresh Q-table; run many episodes with random starting states so the policy
# converges well even for corner states far from the terminal.
Q = collections.defaultdict(lambda: collections.defaultdict(float))

num_optimal_episodes = 5000
for episode in range(num_optimal_episodes):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=2048):
        old_value = Q[curr_state][curr_action]
        next_max  = max(Q[next_state][a] for a in simulator.get_actions(next_state))
        Q[curr_state][curr_action] = next_reward + gamma * next_max
        if next_reward > 0:
            break   # terminal reached; update happened, now cut off for speed

total_states = simulator.resolution * simulator.resolution * 8

with open("policy-optimal.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["state", "action"])
    for state in range(total_states):
        actions = simulator.get_actions(state)
        # Sort by Q value descending; skip actions that self-loop
        sorted_acts = sorted(actions, key=lambda a: Q[state][a], reverse=True)
        chosen = sorted_acts[0]
        for a in sorted_acts:
            _, ns = simulator.get_next_reward_state(state, a)
            if ns != state:
                chosen = a
                break
        writer.writerow([state, chosen])

print(f"Saved {total_states} states to policy-optimal.tsv ({num_optimal_episodes} training episodes)")


Saved 2048 states to policy-optimal.tsv (5000 training episodes)


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

Followed example notebooks provided by the professor in 704 examples folder. I also have a code linter in VS Code to clean up my code.

*Followed example notebooks provided by the professor in 704 examples folder. I also have a code linter in VS Code to clean up my code.*